# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset metadata is described using the [Croissant schema](https://mlcommons.org/croissant/), which ensures rich and reproducible metadata for scientific datasets.


In [ ]:
# Ensure `mlcroissant` is installed. If running in Colab or a new environment, uncomment below:
!pip install mlcroissant

## 1. Data Loading
Load the dataset using the `mlcroissant` library.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset schema URL (Croissant schema)
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata and dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic metadata
print(f"Title: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets and their fields. Use the `@id` (unique identifier) for each. This will help us choose which data to extract for analysis. To interact with data in Croissant, we reference record sets, fields, and columns by their `@id`.

**Note:** We'll programmatically enumerate record sets, list their `@id`s and field `@id`s, and examine a few sample records from each set.

In [ ]:
# List all record sets annotated in the metadata
record_set_infos = []
for rs in getattr(metadata, 'record_set', []):
    rs_id = getattr(rs, '@id', None)
    rs_name = getattr(rs, 'name', None)
    # Get associated fields for this record set
    field_ids = []
    for field in getattr(rs, 'field', []):
        f_id = getattr(field, '@id', None)
        f_name = getattr(field, 'name', None)
        field_ids.append({'@id': f_id, 'name': f_name})
    record_set_infos.append({'@id': rs_id, 'name': rs_name, 'fields': field_ids})

# Print information about record sets
print("Available Record Sets and Their Fields:")
for rs in record_set_infos:
    print(f"Record set '@id': {rs['@id']}, name: {rs['name']}")
    for f in rs['fields']:
        print(f"  Field '@id': {f['@id']}, name: {f['name']}")

# Store all record set @id's for later extraction
record_set_ids = [rs['@id'] for rs in record_set_infos]

# Optionally, print a preview of records for the first record set (if any exist)
if record_set_ids:
    first_record_set = record_set_ids[0]
    print(f"\nSample records from {first_record_set}:")
    for i, rec in enumerate(dataset.records(record_set=first_record_set)):
        print(rec)
        if i > 2:
            break


## 3. Data Extraction
Now, let's load data from specific record sets into Pandas DataFrames for analysis. We'll use the record set and field `@id`s identified above.

In [ ]:
# Extract data from all available record sets
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if not records:
        continue
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"\nLoaded {len(records)} records for record set @id: {record_set_id}")
    print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
    print(dataframes[record_set_id].head())

# Choose primary record set for following analysis
if record_set_ids:
    main_record_set_id = record_set_ids[0]
else:
    main_record_set_id = None


## 4. Exploratory Data Analysis (EDA)
Let's perform some common data processing and transformation steps using one record set. Adjust field `@id`s as necessary based on your data.
- We'll select a numeric field for filtering, normalization, or grouping.
- All field and column names used are their `@id` as listed in the Croissant schema.

In [ ]:
import numpy as np

# Choose a numeric field by @id (adjust as appropriate for your schema)
# For demonstration, we'll pick from the first dataframe's columns
if main_record_set_id and main_record_set_id in dataframes and len(dataframes[main_record_set_id]) > 0:
    df = dataframes[main_record_set_id]
    print(f"All available columns in this record set: {df.columns.tolist()}")
    # Heuristically select the first numeric column, or ask user to set
    numeric_col = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_col = col
            break
    if not numeric_col:
        # Try to coerce columns to numeric
        for col in df.columns:
            try:
                _ = pd.to_numeric(df[col].dropna().iloc[0])
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if df[col].notnull().sum() > 0:
                    numeric_col = col
                    break
            except:
                continue

    if numeric_col:
        print(f"Using numeric column '@id': {numeric_col}")

        # Drop missing values for this demonstration
        df = df[df[numeric_col].notnull()]
        threshold = df[numeric_col].mean()
        filtered_df = df[df[numeric_col] > threshold]
        print(f"Filtered records with {numeric_col} > {threshold:.2f} (mean value):")
        print(filtered_df.head())

        # Normalize the field (z-score)
        filtered_df[f"{numeric_col}_normalized"] = (filtered_df[numeric_col] - filtered_df[numeric_col].mean()) / filtered_df[numeric_col].std()
        print(f"Normalized {numeric_col} for filtered records:")
        print(filtered_df[[numeric_col, f"{numeric_col}_normalized"]].head())

        # Try grouping by another field, e.g. the first non-numeric column
        group_field = None
        for col in df.columns:
            if col != numeric_col and not np.issubdtype(df[col].dtype, np.number):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped by '{group_field}':")
            print(grouped_df.head())
    else:
        print("No numeric field found in this record set for analysis.")
else:
    print("No record set data available for EDA!")

## 5. Visualization
Visualize the distribution of a numeric field or the relationship between two variables using matplotlib or seaborn. Here we show a basic histogram and scatter plot (if sufficient numeric fields exist).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Reuse numeric_col and group_field when possible
if main_record_set_id and main_record_set_id in dataframes and numeric_col and not dataframes[main_record_set_id][numeric_col].isnull().all():
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_col].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_col}")
    plt.xlabel(numeric_col)
    plt.show()

    # If another numeric field exists, do a scatter plot
    second_numeric = None
    for col in df.columns:
        if col != numeric_col and np.issubdtype(df[col].dtype, np.number):
            second_numeric = col
            break
    if second_numeric:
        plt.figure(figsize=(6,6))
        sns.scatterplot(x=df[numeric_col], y=df[second_numeric])
        plt.xlabel(numeric_col)
        plt.ylabel(second_numeric)
        plt.title(f"{numeric_col} vs {second_numeric}")
        plt.show()
    elif group_field:
        # Boxplot by group
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field], y=df[numeric_col])
        plt.title(f"{numeric_col} grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Insufficient numeric data for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load a Croissant-annotated dataset with `mlcroissant` using its schema URL;
- Explore record sets and fields (using their `@id` for reference);
- Load record sets into pandas DataFrames and perform basic EDA using only the `@id`s;
- Visualize key numeric variables from the dataset.

This approach enables reproducible, schema-driven workflows for multi-modal FAIR datasets. You can now further process, analyze, or model from these dataframes using your favorite Python/ML tools.